# Fine-tuning LLM's with LoRa & QLoRa
<a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/llm-fine-tuning-lora-qlora/llm-fine-tune-tutorial.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>




Full code and repo:

https://github.com/unionai/workshops/tree/main/tutorials/llm-fine-tuning-lora-qlora

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    #setup project and keyring if in colab env
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/llm-fine-tuning-lora-qlora
    !uv pip install -r requirements.txt
    !uv pip install -r requirements.txt keyrings.alt
    !mkdir -p ~/.config/python_keyring && echo -e "[backend]\ndefault-keyring=keyrings.alt.file.PlaintextKeyring" > ~/.config/python_keyring/keyringrc.cfg
    %env TERM=dumb

In [ ]:
!flyte create config \
    --endpoint tryv2.hosted.unionai.cloud \
    --project workshoplora \
    --domain development \
    --builder remote \
    --auth-type headless

## Run the fine-tuning pipeline

##### Pipeline Parameters

| Flag | Default | Description |
|------|---------|-------------|
| `--model_name` | `HuggingFaceTB/SmolLM2-135M` | HuggingFace model to fine-tune |
| `--dataset_name` | `b-mc2/sql-create-context` | HuggingFace dataset |
| `--method` | `lora` | Fine-tuning method: `full`, `lora`, or `qlora` |
| `--epochs` | `3` | Training epochs |
| `--lr` | `2e-4` | Learning rate |
| `--batch_size` | `4` | Per-device batch size |
| `--max_train_samples` | `5000` | Max training examples |
| `--max_eval_samples` | `500` | Max evaluation examples |
| `--num_eval_examples` | `50` | Examples for before/after comparison |
| `--lora_r` | `16` | LoRA rank (for lora/qlora) |
| `--lora_alpha` | `32` | LoRA alpha (for lora/qlora) |

In [ ]:
!flyte run workflow.py pipeline --method lora --epochs 3 --max_train_samples 5000 --num_eval_examples 500

You can also run this locally without docker or kubernetes by just adding the `--local` flag
```
flyte run --local workflow.py pipeline --method lora --epochs 3 --max_train_samples 5000 --num_eval_example 500
```

Or setup a local Flyte devbox that uses docker 

```
flyte start devbox
```

## Serve the Model

You need to wait until the training pipeline is complete before you serve the fine-tuned model

In [ ]:
# !python serve.py # Will get latest model

!python serve.py --run-name rk2zfpk6x49c5vs45652 # gets model by run ID

In [ ]:
# change hosted model URL when you deploy your own

%%curl -X POST https://steep-fog-ad9c2.apps.tryv2.hosted.unionai.cloud/generate \
  -H "Content-Type: application/json" \
  -d '{
    "schema": "CREATE TABLE employees (id INT, name VARCHAR, department VARCHAR, salary INT)",
    "question": "What is the average salary by department?"
  }'

# Serve Gradio UI for Model

In [ ]:
!python app_gradio.py

Want to do more?

- Get Started with Flyte: https://www.union.ai/docs/v2/flyte/user-guide/run-modes/
- Book a consultation: https://www.union.ai/consultation
- Join the Slack: https://slack.flyte.org/